# Weather Data Pipeline (Legacy)

**Source script:** `weather_data_final.py`

Notebook mirror for submission traceability.

In [ ]:
import time
import json
import hashlib
import re
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import deque




EXCEL_PATH = "Data_set_cats for analysis.xlsx"
SHEET_NAME = "Tabelle1"

ZIP_COL = "zip code"
DATE_COL = "date of presentation"
CAT_ID_COL = "ID"

LOOKBACK_DAYS = 14
COUNTRY_CODE = "DE"
TIMEZONE = "Europe/Berlin"
GERMANY_BOUNDS = {
    "lat_min": 47.0,
    "lat_max": 55.2,
    "lon_min": 5.5,
    "lon_max": 15.5,
}

CACHE_DIR = Path("cache_openmeteo")
RAW_DIR = CACHE_DIR / "raw"
FEATURE_DIR = CACHE_DIR / "features"
CACHE_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)
FEATURE_DIR.mkdir(exist_ok=True)

MANIFEST_PATH = CACHE_DIR / "manifest.csv"
CATS_TO_JOB_PATH = CACHE_DIR / "cats_to_weather_job.csv"
GEOCODE_CACHE_PATH = CACHE_DIR / "zip_geocode_cache.json"
LOCAL_GEOCODE_PATH = Path("geocoded_locations.csv")

OPEN_METEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
OPEN_METEO_GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"

DAILY_VARS = [
    "temperature_2m_max","temperature_2m_min","temperature_2m_mean",
    "apparent_temperature_max","apparent_temperature_min","apparent_temperature_mean",
    "sunrise","sunset","daylight_duration","sunshine_duration",
    "precipitation_sum","precipitation_hours","rain_sum","snowfall_sum",
    "shortwave_radiation_sum",
    "wind_speed_10m_max","wind_gusts_10m_max","wind_direction_10m_dominant",
    "et0_fao_evapotranspiration",
    "uv_index_max","surface_pressure_max","moon_phase"
]

FALLBACK_DAILY_VARS = [
    "temperature_2m_max","temperature_2m_min","temperature_2m_mean",
    "apparent_temperature_max","apparent_temperature_min","apparent_temperature_mean",
    "sunrise","sunset","daylight_duration","sunshine_duration",
    "precipitation_sum","precipitation_hours","rain_sum","snowfall_sum",
    "shortwave_radiation_sum",
    "wind_speed_10m_max","wind_gusts_10m_max","wind_direction_10m_dominant",
    "et0_fao_evapotranspiration"
]

RATE_LIMITS = {
    "per_min": 600,
    "per_hour": 5000,
    "per_day": 10000,
}




class SlidingWindowRateLimiter:
    def __init__(self, per_min, per_hour, per_day):
        self.calls_60 = deque()
        self.calls_3600 = deque()
        self.calls_86400 = deque()
        self.per_min = per_min
        self.per_hour = per_hour
        self.per_day = per_day

    def _prune(self, now):
        while self.calls_60 and now - self.calls_60[0] >= 60:
            self.calls_60.popleft()
        while self.calls_3600 and now - self.calls_3600[0] >= 3600:
            self.calls_3600.popleft()
        while self.calls_86400 and now - self.calls_86400[0] >= 86400:
            self.calls_86400.popleft()

    def wait(self):
        logged = False
        while True:
            now = time.time()
            self._prune(now)
            if (len(self.calls_60) < self.per_min and
                len(self.calls_3600) < self.per_hour and
                len(self.calls_86400) < self.per_day):
                if logged:
                    log("Rate limiter released; continuing requests.")
                return
            if not logged:
                log("Rate limiter engaged; waiting before next request.")
                logged = True
            time.sleep(5)

    def record(self):
        now = time.time()
        self.calls_60.append(now)
        self.calls_3600.append(now)
        self.calls_86400.append(now)




class NonRetryableError(Exception):
    pass

def log(msg, level="INFO"):
    ts = datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")
    print(f"[{ts}] {level}: {msg}", flush=True)

def load_json(path):
    if path.exists():
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError as e:
            log(f"Failed to parse JSON cache {path}: {e}. Starting fresh.", level="WARN")
            return {}
    return {}

def save_json(path, obj):
    path.write_text(json.dumps(obj, indent=2))

def make_weather_job_id(zip_code, start_date, end_date):
    raw = f"{zip_code}|{start_date}|{end_date}|daily_v1"
    return hashlib.sha1(raw.encode()).hexdigest()[:12]

def parse_invalid_daily_var(message):
    match = re.search(r"invalid String value ([A-Za-z0-9_]+)", message)
    if match:
        return match.group(1)
    return None

def is_in_germany(lat, lon):
    return (
        GERMANY_BOUNDS["lat_min"] <= lat <= GERMANY_BOUNDS["lat_max"]
        and GERMANY_BOUNDS["lon_min"] <= lon <= GERMANY_BOUNDS["lon_max"]
    )

def load_local_geocode(path):
    if not path.exists():
        log(f"Local geocode file not found at {path}; falling back to API.", level="WARN")
        return {}
    try:
        df = pd.read_csv(path, dtype={"zip_code": str})
    except Exception as e:
        log(f"Failed to read local geocode file {path}: {e}.", level="WARN")
        return {}
    missing = [c for c in ("zip_code", "latitude", "longitude") if c not in df.columns]
    if missing:
        log(f"Local geocode file missing columns {missing}; ignoring.", level="WARN")
        return {}
    df["zip_code"] = (
        df["zip_code"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(5)
    )
    df = df.dropna(subset=["zip_code", "latitude", "longitude"])
    mapping = {}
    for _, row in df.iterrows():
        zip_code = row["zip_code"]
        if zip_code not in mapping:
            mapping[zip_code] = (float(row["latitude"]), float(row["longitude"]))
    log(f"Loaded {len(mapping)} local geocodes from {path}.")
    return mapping

def request_json(url, params, label, timeout):
    log(f"{label}: requesting {url} with params={params}")
    try:
        r = requests.get(url, params=params, timeout=timeout)
    except requests.RequestException as e:
        raise RuntimeError(f"{label} request failed: {e}") from e
    status = r.status_code
    if status >= 400:
        message = ""
        try:
            payload = r.json()
            message = payload.get("reason") or payload.get("error") or ""
        except ValueError:
            message = (r.text or "").strip()
        message = message.replace("\n", " ")[:200]
        if status == 429 or 500 <= status < 600:
            raise RuntimeError(f"{label} HTTP {status}: {message}")
        raise NonRetryableError(f"{label} HTTP {status}: {message}")
    try:
        data = r.json()
    except ValueError as e:
        snippet = (r.text or "").strip().replace("\n", " ")[:200]
        raise NonRetryableError(f"{label} returned non-JSON response: {snippet}") from e
    log(f"{label}: response received with keys={list(data.keys())}.")
    return data




def build_manifest():
    log(f"Loading spreadsheet {EXCEL_PATH} (sheet={SHEET_NAME}).")
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

    missing_cols = [c for c in (CAT_ID_COL, ZIP_COL, DATE_COL) if c not in df.columns]
    if missing_cols:
        raise NonRetryableError(f"Missing required columns: {missing_cols}")

    raw_rows = len(df)
    zip_clean = df[ZIP_COL].astype(str).str.replace(r"\D", "", regex=True)
    zip_clean = zip_clean.where(zip_clean.str.len() > 0)
    zip_clean = zip_clean.str.zfill(5)
    df[ZIP_COL] = zip_clean

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.date
    invalid_zip = df[ZIP_COL].isna() | (df[ZIP_COL].str.len() != 5)
    invalid_date = df[DATE_COL].isna()
    invalid_rows = invalid_zip | invalid_date
    if invalid_rows.any():
        bad_count = int(invalid_rows.sum())
        log(f"Dropping {bad_count} rows with invalid ZIP/date values.", level="WARN")
        df = df.loc[~invalid_rows].copy()

    log(f"Prepared {len(df)} valid rows (from {raw_rows}).")

    jobs = []
    manifest_cols = [
        "weather_job_id","zip","start_date","end_date","status","attempts",
        "last_error","last_try_ts","raw_path","feature_path"
    ]
    for zip_code, sub in df.groupby(ZIP_COL):
        min_d = min(sub[DATE_COL])
        max_d = max(sub[DATE_COL])
        start = min_d - timedelta(days=LOOKBACK_DAYS)
        end = max_d
        job_id = make_weather_job_id(zip_code, start, end)

        jobs.append({
            "weather_job_id": job_id,
            "zip": zip_code,
            "start_date": str(start),
            "end_date": str(end),
            "status": "PENDING",
            "attempts": 0,
            "last_error": "",
            "last_try_ts": "",
            "raw_path": RAW_DIR / f"{zip_code}_{start}_{end}.parquet",
            "feature_path": FEATURE_DIR / f"{zip_code}_{start}_{end}.parquet",
        })

    if jobs:
        manifest = pd.DataFrame(jobs)
    else:
        manifest = pd.DataFrame(columns=manifest_cols)

    if MANIFEST_PATH.exists():
        old = pd.read_csv(MANIFEST_PATH, dtype={"zip": str})
        if old.columns.duplicated().any():
            log("Old manifest has duplicate columns; de-duplicating.", level="WARN")
            old = old.loc[:, ~old.columns.duplicated()]
        old = old[[c for c in old.columns if not c.endswith("_old")]]
        if "weather_job_id" in old.columns:
            old = old.set_index("weather_job_id")
            manifest = manifest.set_index("weather_job_id")
            for col in ["status","attempts","last_error","last_try_ts","raw_path","feature_path"]:
                if col in old.columns:
                    manifest[col] = old[col].combine_first(manifest[col])
            manifest = manifest.reset_index()
        else:
            log("Old manifest missing weather_job_id; skipping merge.", level="WARN")

    manifest["attempts"] = pd.to_numeric(manifest["attempts"], errors="coerce").fillna(0).astype(int)
    manifest["raw_path"] = manifest["raw_path"].astype(str)
    manifest["feature_path"] = manifest["feature_path"].astype(str)
    manifest.to_csv(MANIFEST_PATH, index=False)
    log(f"Wrote manifest with {len(manifest)} jobs to {MANIFEST_PATH}.")

    df["weather_job_id"] = df[ZIP_COL].map(
        manifest.set_index("zip")["weather_job_id"]
    )

    df[[CAT_ID_COL, ZIP_COL, DATE_COL, "weather_job_id"]]\
        .rename(columns={ZIP_COL:"zip", DATE_COL:"date_of_presentation"})\
        .to_csv(CATS_TO_JOB_PATH, index=False)
    log(f"Wrote cat-to-job mapping to {CATS_TO_JOB_PATH}.")

    return manifest




def geocode_zip(zip_code, cache, local_cache=None):
    key = f"{COUNTRY_CODE}:{zip_code}"
    if local_cache and zip_code in local_cache:
        lat, lon = local_cache[zip_code]
        if not is_in_germany(lat, lon):
            log(
                f"Local geocode for {zip_code} is outside Germany bounds; ignoring.",
                level="WARN"
            )
        else:
            cache[key] = {"lat": lat, "lon": lon}
            save_json(GEOCODE_CACHE_PATH, cache)
            log(f"Local geocode hit for {zip_code} (lat={lat}, lon={lon}).")
            return lat, lon
    if key in cache:
        lat = cache[key].get("lat")
        lon = cache[key].get("lon")
        if lat is not None and lon is not None and is_in_germany(lat, lon):
            log(f"Geocode cache hit for {zip_code}.")
            return lat, lon
        log(f"Cached geocode for {zip_code} is invalid/outside Germany; refreshing.", level="WARN")

    query_params = [
        {
            "name": zip_code,
            "count": 10,
            "format": "json",
            "country": COUNTRY_CODE,
            "country_code": COUNTRY_CODE
        },
        {
            "name": f"{zip_code} {COUNTRY_CODE}",
            "count": 10,
            "format": "json"
        },
        {
            "name": f"{zip_code} Germany",
            "count": 10,
            "format": "json"
        }
    ]

    res = None
    for attempt, params in enumerate(query_params, start=1):
        data = request_json(OPEN_METEO_GEOCODE_URL, params, f"Geocode[{attempt}]", timeout=30)
        results = data.get("results") or []
        if not results:
            log(f"Geocode attempt {attempt} returned no results for {zip_code}.", level="WARN")
            continue
        matches = [r for r in results if r.get("country_code") == COUNTRY_CODE]
        if matches:
            res = matches[0]
            break
        log(
            f"Geocode attempt {attempt} returned {len(results)} results but none in {COUNTRY_CODE}.",
            level="WARN"
        )

    if res is None:
        reason = data.get("reason") or data.get("error") or "no results in response"
        raise NonRetryableError(f"Geocode returned no {COUNTRY_CODE} results for {zip_code}: {reason}")
    if "latitude" not in res or "longitude" not in res:
        raise NonRetryableError(f"Geocode response missing coordinates for {zip_code}.")
    cache[key] = {"lat": res["latitude"], "lon": res["longitude"]}
    save_json(GEOCODE_CACHE_PATH, cache)
    log(
        f"Geocoded {zip_code} to lat={res['latitude']} lon={res['longitude']} "
        f"(country={res.get('country_code','?')})."
    )
    return res["latitude"], res["longitude"]

def fetch_daily(lat, lon, start, end):
    base_params = [
        ("latitude", lat),
        ("longitude", lon),
        ("start_date", start),
        ("end_date", end),
        ("timezone", TIMEZONE),
    ]

    vars_to_try = list(DAILY_VARS)
    tried_fallback = False

    while True:
        params = base_params + [("daily", var) for var in vars_to_try]
        try:
            data = request_json(OPEN_METEO_ARCHIVE_URL, params, "Archive", timeout=60)
            break
        except NonRetryableError as e:
            msg = str(e)
            if "ForecastVariableDaily" in msg or "invalid String value" in msg:
                log(f"Archive rejected daily list: {msg}", level="WARN")
                bad = parse_invalid_daily_var(msg)
                if bad and bad in vars_to_try:
                    log(f"Archive rejected daily var '{bad}'; removing and retrying.", level="WARN")
                    vars_to_try = [v for v in vars_to_try if v != bad]
                    if not vars_to_try:
                        raise NonRetryableError("All daily variables were rejected by the archive API.")
                    continue
                if not tried_fallback:
                    log("Archive daily list rejected; switching to fallback daily variable set.", level="WARN")
                    vars_to_try = list(FALLBACK_DAILY_VARS)
                    tried_fallback = True
                    continue
            raise
        except Exception:
            raise

    if vars_to_try != DAILY_VARS:
        log(f"Archive using {len(vars_to_try)} daily variables after filtering.")
    if "daily" not in data or not data["daily"]:
        reason = data.get("reason") or data.get("error") or "missing daily data"
        raise NonRetryableError(f"Archive response missing daily data: {reason}")
    df = pd.DataFrame(data["daily"])
    df["time"] = pd.to_datetime(df["time"]).dt.date
    log(f"Fetched {len(df)} daily rows for {start} to {end}.")
    return df.set_index("time")

def build_features(df):
    log(f"Building features from {len(df)} daily rows.")
    out = pd.DataFrame(index=df.index)
    day1_only = {"sunrise","sunset","moon_phase"}

    for col in df.columns:
        s = df[col]
        out[f"{col}__day1"] = s
        if col in day1_only:
            continue
        out[f"{col}__mean_2d"] = s.rolling(2).mean()
        out[f"{col}__mean_3d"] = s.rolling(3).mean()
        out[f"{col}__mean_7d"] = s.rolling(7).mean()
        out[f"{col}__mean_13d"] = s.rolling(13).mean()
        out[f"{col}__peak_7d"] = s.rolling(7).max()
        out[f"{col}__peak_14d"] = s.rolling(14).max()

    result = out.reset_index()
    log(f"Built {len(result.columns)} feature columns.")
    return result




def with_retries(fn, label="job"):
    delays = [0, 60, 600]
    last_err = None
    for attempt, delay in enumerate(delays, start=1):
        if delay:
            log(f"{label}: waiting {delay}s before attempt {attempt}.")
            time.sleep(delay)
        try:
            return fn()
        except NonRetryableError as e:
            log(f"{label}: non-retryable error: {e}", level="ERROR")
            raise
        except Exception as e:
            last_err = e
            log(f"{label}: attempt {attempt} failed: {e}", level="WARN")
    raise last_err




def run_weather_pipeline():
    log("Starting weather pipeline.")
    manifest = build_manifest()
    geo_cache = load_json(GEOCODE_CACHE_PATH)
    local_geo = load_local_geocode(LOCAL_GEOCODE_PATH)
    limiter = SlidingWindowRateLimiter(**RATE_LIMITS)
    log(f"Loaded geocode cache with {len(geo_cache)} entries.")
    log(f"Loaded local geocode cache with {len(local_geo)} entries.")
    log(f"Manifest contains {len(manifest)} jobs.")

    if manifest.empty:
        log("Manifest is empty; nothing to do.", level="WARN")
        return

    total_jobs = len(manifest)
    for i, row in manifest.iterrows():
        log(f"Processing job {i + 1}/{total_jobs} (zip {row['zip']}).")
        if row["status"] == "DONE":
            log(f"Skipping completed job {row['weather_job_id']} (zip {row['zip']}).")
            continue

        manifest.at[i, "last_try_ts"] = datetime.now(timezone.utc).isoformat()
        manifest.at[i, "attempts"] = int(manifest.at[i, "attempts"]) + 1
        manifest.to_csv(MANIFEST_PATH, index=False)

        try:
            def job():
                log(f"Job {row['weather_job_id']}: fetching {row['zip']} {row['start_date']} to {row['end_date']}.")
                start_ts = time.time()
                limiter.wait()
                limiter.record()
                lat, lon = geocode_zip(row["zip"], geo_cache, local_geo)
                daily = fetch_daily(lat, lon, row["start_date"], row["end_date"])
                daily.to_parquet(row["raw_path"])
                log(f"Saved raw daily data to {row['raw_path']}.")
                feats = build_features(daily)
                feats.to_parquet(row["feature_path"])
                log(f"Saved features to {row['feature_path']}.")
                log(f"Job {row['weather_job_id']} complete in {time.time() - start_ts:.1f}s.")
            with_retries(job, label=f"job {row['weather_job_id']}")

            manifest.at[i, "status"] = "DONE"
            manifest.at[i, "last_error"] = ""

        except Exception as e:
            manifest.at[i, "status"] = "FAILED"
            manifest.at[i, "last_error"] = repr(e)
            log(f"Job {row['weather_job_id']} failed: {e}", level="ERROR")

        manifest.to_csv(MANIFEST_PATH, index=False)
        log(f"Updated manifest for job {row['weather_job_id']} with status {manifest.at[i, 'status']}.")

    log("Weather pipeline complete (resumable).")


if __name__ == "__main__":
    run_weather_pipeline()
